In [11]:
import warnings
from rdkit import RDLogger

# 屏蔽 RDKit 警告
RDLogger.DisableLog('rdApp.*')

# 或屏蔽所有 Python 警告
warnings.filterwarnings("ignore")
# 屏蔽 LightGBM 警告
warnings.filterwarnings("ignore", category=UserWarning, module="lightgbm")

In [12]:
import torch
from sklearn.model_selection import StratifiedKFold
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.metrics import precision_recall_curve, auc
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import joblib
import optuna
from rdkit.Chem import Descriptors, AllChem
from tqdm import tqdm  # 导入tqdm
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupKFold





In [13]:
# 函数：将SMILES转换为分子描述符和指纹
def smiles_to_features(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    # 提取描述符
    descriptors = [
        Descriptors.MolWt(mol),  # 分子量
        Descriptors.MolLogP(mol),  # LogP
        Descriptors.NumHDonors(mol),  # 氢键供体数量
        Descriptors.NumHAcceptors(mol)  # 氢键受体数量
    ]
    # 生成Morgan指纹
    fingerprint = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048)
    fingerprint_array = np.zeros((2048,))
    Chem.DataStructs.ConvertToNumpyArray(fingerprint, fingerprint_array)
    # 合并描述符和指纹
    features = np.concatenate([descriptors, fingerprint_array])
    return features


In [14]:
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold
from tqdm import tqdm
import optuna
import numpy as np

def train_evaluate_regression_model_with_optuna(model_name, model_class, param_func, X, y, groups):
    def objective(trial):
        params = param_func(trial)
        model = model_class(**params)

        gkf = GroupKFold(n_splits=10)
        maes = []

        for train_idx, val_idx in tqdm(gkf.split(X, y, groups=groups), total=10, desc=f"Training {model_name}"):
            X_train, X_val = X[train_idx], X[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]

            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)

            # ✅ 计算 MAE
            mae = mean_absolute_error(y_val, y_pred)
            maes.append(mae)

        return np.mean(maes)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=30)

    print(f'Best parameters for {model_name}: {study.best_params}')
    print(f'Best mean MAE: {study.best_value:.4f}')
    

In [15]:
# 数据预处理
df = pd.read_excel('../algae_unique.xlsx')
labels = df['mgperL'].values
smiles_list = df['SMILES_Canonical_RDKit'].tolist()
endpoints_a = df['endpoint']
Duration_Values_a = df['Duration_Value'].values
effects_a = df['effect']


In [16]:

features = []
new_labels = []
new_smiles_list = []
endpoints = []
Duration_Values = []
effects =[]


for smiles, label,a,b,c in zip(smiles_list, labels,Duration_Values_a,effects_a,endpoints_a):
    feature = smiles_to_features(smiles)
    if feature is not None:
        features.append(feature)
        new_labels.append(label)
        new_smiles_list.append(smiles)
        Duration_Values.append(a)
        effects.append(b)
        endpoints.append(c)

X = np.array(features)
y = np.array(new_labels)
groups = new_smiles_list  # 可直接用于 GroupKFold




In [17]:
def encode_column(zz):
    zz_series = pd.Series(zz)  # 转换为 Series
    unique_values = zz_series.unique()
    if len(unique_values) > 1:
        encoder = OneHotEncoder(sparse_output=False)
        return encoder.fit_transform(zz_series.values.reshape(-1, 1))
    else:
        return None  # 只有一种类别时忽略

Duration_Values =pd.Series(Duration_Values)


# 编码 effect、endpoint 和 species_group 列
effect_encoded = encode_column(effects)
endpoint_encoded = encode_column(endpoints)
#species_encoded = encode_column(df, 'species_group')

# # 将需要的列拼接成输入 X
X = np.hstack((X, Duration_Values.values.reshape(-1, 1)))

# # 拼接编码后的列（如果存在）
for encoded_feature in [effect_encoded, endpoint_encoded]:
     if encoded_feature is not None:
         X = np.hstack((X, encoded_feature))



y=np.log1p(y)

In [8]:
def xgb_param_func(trial):
    return {
        'n_estimators': trial.suggest_int('n_estimators', 100, 600),
        'max_depth': trial.suggest_int('max_depth', 5, 20),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),   # L1 正则
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0)  # L2 正则
    }
from xgboost import XGBRegressor

train_evaluate_regression_model_with_optuna(
    "XGBoost",
    XGBRegressor,
    xgb_param_func,
    X, y, groups
)

[I 2025-05-15 22:03:57,828] A new study created in memory with name: no-name-0a518b80-f6fc-40c0-8cbc-d0c2a896f42a
Training XGBoost: 100%|██████████| 10/10 [01:30<00:00,  9.06s/it]
[I 2025-05-15 22:05:28,418] Trial 0 finished with value: 1.1643489475896451 and parameters: {'n_estimators': 362, 'max_depth': 12, 'learning_rate': 0.016038948469934818, 'subsample': 0.9913643812571765, 'colsample_bytree': 0.6821227171228235, 'reg_alpha': 0.7740812128701673, 'reg_lambda': 0.6438407480527664}. Best is trial 0 with value: 1.1643489475896451.
Training XGBoost: 100%|██████████| 10/10 [01:33<00:00,  9.37s/it]
[I 2025-05-15 22:07:02,103] Trial 1 finished with value: 1.1218832994015222 and parameters: {'n_estimators': 318, 'max_depth': 20, 'learning_rate': 0.11991387627085225, 'subsample': 0.8298367502644318, 'colsample_bytree': 0.7264927864641696, 'reg_alpha': 0.07727246948243749, 'reg_lambda': 0.09864227332519415}. Best is trial 1 with value: 1.1218832994015222.
Training XGBoost: 100%|██████████| 

Best parameters for XGBoost: {'n_estimators': 396, 'max_depth': 20, 'learning_rate': 0.038315190727745016, 'subsample': 0.7041184898957844, 'colsample_bytree': 0.9496590514328553, 'reg_alpha': 0.9964564706169682, 'reg_lambda': 0.4892503870578777}
Best mean MAE: 1.1002


In [9]:
from lightgbm import LGBMRegressor

def lgbm_param_func(trial):
    return {
        'n_estimators': trial.suggest_int('n_estimators', 100, 600),
        'max_depth': trial.suggest_int('max_depth', 5, 20),
        'num_leaves': trial.suggest_int('num_leaves', 20, 300),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0),
        'verbose': -1
    }

print("Training LightGBM (Poisson)...")
train_evaluate_regression_model_with_optuna(
    "LightGBM",
    lambda **params: LGBMRegressor(objective="poisson", **params),  # ✅ 加入 Poisson 目标
    lgbm_param_func,
    X, y, groups
)


[I 2025-05-16 07:57:35,837] A new study created in memory with name: no-name-69d6890c-b020-465a-9bf9-281b7da7d772


Training LightGBM (Poisson)...


Training LightGBM: 100%|██████████| 10/10 [00:17<00:00,  1.70s/it]
[I 2025-05-16 07:57:52,883] Trial 0 finished with value: 1.289686428065331 and parameters: {'n_estimators': 171, 'max_depth': 13, 'num_leaves': 216, 'learning_rate': 0.015029222181771483, 'feature_fraction': 0.7669056505478112, 'bagging_fraction': 0.7065000864603446, 'bagging_freq': 2, 'reg_alpha': 0.8125198950492725, 'reg_lambda': 0.8131241094361006}. Best is trial 0 with value: 1.289686428065331.
Training LightGBM: 100%|██████████| 10/10 [00:12<00:00,  1.28s/it]
[I 2025-05-16 07:58:05,695] Trial 1 finished with value: 1.1294258307704403 and parameters: {'n_estimators': 515, 'max_depth': 8, 'num_leaves': 283, 'learning_rate': 0.08652940711999334, 'feature_fraction': 0.8140883218191595, 'bagging_fraction': 0.6781882248007682, 'bagging_freq': 7, 'reg_alpha': 0.2261144455159032, 'reg_lambda': 0.7687256697124112}. Best is trial 1 with value: 1.1294258307704403.
Training LightGBM: 100%|██████████| 10/10 [00:10<00:00,  1.04s

Best parameters for LightGBM: {'n_estimators': 474, 'max_depth': 20, 'num_leaves': 196, 'learning_rate': 0.19575659362271527, 'feature_fraction': 0.696563175505503, 'bagging_fraction': 0.879334932393794, 'bagging_freq': 4, 'reg_alpha': 0.39492944811226344, 'reg_lambda': 0.6153467373587957}
Best mean MAE: 1.0816


In [18]:
import random

# 固定随机种子
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # for multi-GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)  # 设置固定种子

In [19]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
import optuna
import numpy as np


class DNNWithSoftplus(nn.Module):
    def __init__(self, input_dim, hidden_sizes, activation):
        super().__init__()
        act_fn = {
            'relu': nn.ReLU(),
            'logistic': nn.Sigmoid(),
            'tanh': nn.Tanh()
        }[activation]
        layers = []
        prev_dim = input_dim
        for h in hidden_sizes:
            layers += [nn.Linear(prev_dim, h), act_fn]
            prev_dim = h
        layers += [nn.Linear(prev_dim, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return F.softplus(self.net(x)).squeeze(-1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def train_dnn_with_optuna_pytorch(X, y, groups, device=device):
    def dnn_param_func(trial):
        return {
            'hidden_layer_sizes': trial.suggest_categorical(
                'hidden_layer_sizes', [(50,), (100,), (150,), (100, 50), (150, 100, 50)]
            ),
            'activation': trial.suggest_categorical('activation', ['relu', 'logistic', 'tanh']),
            'alpha': trial.suggest_float('alpha', 1e-5, 1e-2, log=True),
            'learning_rate': trial.suggest_float('learning_rate_init', 1e-4, 1e-2, log=True),
            'optimizer': trial.suggest_categorical('solver', ['adam', 'sgd'])
        }

    def objective(trial):
        params = dnn_param_func(trial)
        model = DNNWithSoftplus(
            input_dim=X.shape[1],
            hidden_sizes=params['hidden_layer_sizes'],
            activation=params['activation']
        ).to(device)

        optimizer = {
            'adam': torch.optim.Adam,
            'sgd': torch.optim.SGD
        }[params['optimizer']](model.parameters(), lr=params['learning_rate'], weight_decay=params['alpha'])

        loss_fn = nn.MSELoss()
        gkf = GroupKFold(n_splits=10)
        fold_maes = []

        for train_idx, val_idx in gkf.split(X, y, groups=groups):
            X_train, y_train = X[train_idx], y[train_idx]
            X_val, y_val = X[val_idx], y[val_idx]

            scaler = StandardScaler()
            X_train = scaler.fit_transform(X_train)
            X_val = scaler.transform(X_val)

            train_ds = TensorDataset(torch.tensor(X_train).float(), torch.tensor(y_train).float())
            train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)

            model.train()
            for epoch in range(100):
                for xb, yb in train_loader:
                    xb, yb = xb.to(device), yb.to(device)
                    optimizer.zero_grad()
                    pred = model(xb)
                    loss = loss_fn(pred, yb)
                    loss.backward()
                    optimizer.step()

            model.eval()
            with torch.no_grad():
                val_preds = model(torch.tensor(X_val).float().to(device)).cpu().numpy()
                mae = mean_absolute_error(y_val, val_preds)
                fold_maes.append(mae)

        return np.mean(fold_maes)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=30)
    print("\n✅ Best Parameters Found:")
    print(study.best_params)
    print(f"Mean MAE = {study.best_value:.4f}")
    return study.best_params


best_dnn_params = train_dnn_with_optuna_pytorch(X, y, groups)

[I 2025-05-16 12:57:54,226] A new study created in memory with name: no-name-1184640e-3a3b-4161-a022-2b9d29026d54
[I 2025-05-16 13:00:51,958] Trial 0 finished with value: 1.251310085880619 and parameters: {'hidden_layer_sizes': (50,), 'activation': 'tanh', 'alpha': 2.068678830209306e-05, 'learning_rate_init': 0.00919380741907174, 'solver': 'adam'}. Best is trial 0 with value: 1.251310085880619.
[I 2025-05-16 13:03:09,881] Trial 1 finished with value: 0.8918952343759982 and parameters: {'hidden_layer_sizes': (100, 50), 'activation': 'tanh', 'alpha': 5.199036828865284e-05, 'learning_rate_init': 0.00028478786892784656, 'solver': 'sgd'}. Best is trial 1 with value: 0.8918952343759982.
[I 2025-05-16 13:05:45,636] Trial 2 finished with value: 1.3548083770575405 and parameters: {'hidden_layer_sizes': (150,), 'activation': 'tanh', 'alpha': 0.005599980608322632, 'learning_rate_init': 0.009268534607830557, 'solver': 'adam'}. Best is trial 1 with value: 0.8918952343759982.
[I 2025-05-16 13:08:04,


✅ Best Parameters Found:
{'hidden_layer_sizes': (150,), 'activation': 'relu', 'alpha': 0.0005422346518613921, 'learning_rate_init': 0.009343476386401628, 'solver': 'sgd'}
Mean MAE = 0.5408
